# voiceai — Echter Mini-Trainingslauf (1 Zelle, 1 Play)

**Saubere Architektur:** eigenes frisches **uv-venv mit eigenem Python**.
Wir fassen Colabs vorinstallierte Pakete gar nicht an → keine Versionskonflikte,
kein torchvision/torchao-Gefrickel, nie wieder Dependency-Probleme.

**3 Schritte:**
1. Oben: **Runtime → Change runtime type → T4 GPU** → Save
2. In der Zelle unten deinen Hugging-Face-Token einfügen (https://huggingface.co/settings/tokens)
3. **Play** drücken. Fertig.

Erster Lauf lädt ein frisches Python + Torch ins venv (~3-5 min). Danach echtes
Training: 64 echte LibriSpeech-Clips → Mimi → Qwen3.5-0.8B Adapter, Loss sinkt.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'  # <-- nur das hier aendern
assert os.environ['HF_TOKEN'].startswith('hf_'), 'HF-Token noch nicht eingetragen!'

# 1) uv holen (per curl, kein pip)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# 2) Repo
![ -d voiceai ] || git clone -q https://github.com/chukfinley/voiceai.git
%cd /content/voiceai
!git pull -q

# 3) FRISCHES isoliertes venv mit eigenem Python — Colabs Pakete bleiben unberuehrt
!~/.local/bin/uv venv --python 3.12 --seed
!~/.local/bin/uv pip install -e '.[train,dev]'

# 4) GPU-Check IM venv (eigenes Torch, nicht Colabs)
!.venv/bin/python -c "import torch; assert torch.cuda.is_available(), 'Keine GPU! Runtime -> T4 GPU'; print('GPU:', torch.cuda.get_device_name(0))"

# 5) echter Mini-Trainingslauf, aus dem venv
!.venv/bin/python scripts/colab_realtest.py

Am Ende: `[done] adapters -> runs/realtest/` und der Loss in der Fortschrittsleiste
ist über die 200 Schritte gefallen.

Zu wenig GPU-Speicher? Letzte Zeile ersetzen durch:
`!.venv/bin/python scripts/colab_realtest.py --backbone Qwen/Qwen3-0.6B --steps 150`